# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 제출 정보 (Chapter 06 제출 답안)

- 이름: 장태진
- GitHub ID: Taejin-AI
- 개인 저장소: `llm-data-analysis-study`
- 작성일: 2026.10.22
- 사용한 LLM: Claude (Anthropic)
- 실행 환경: Windows 11, 강사 저장소 공식 clone(`https://github.com/GilbertMoon/llm-data-analysis-course`, HEAD `6105f42`), 프로젝트 `.venv`(Python 3.13.1, pandas 3.0.5) 커널. 실제 경로는 4번 셀 Output에 있습니다.
- 최종 Notebook URL:

```text
https://github.com/Taejin-AI/llm-data-analysis-study/blob/master/chapter06/chapter06.ipynb
```

> 작성 방식: 공식 Notebook `notebooks/ch06_eda_questions.ipynb`를 복사해 1~23번 셀을 순서대로 그대로 실행했습니다. 실습 가이드 STEP의 Evidence와 답안 양식 항목은 `[추가 검증]` 코드 셀과 Markdown 셀로 사이에 추가했습니다. 공식 셀 중 수정한 것은 두 개입니다. 4번 경로 탐색 셀과, 고객 이름(`name`)을 출력하던 14번 고객별 집계 셀이며, 둘 다 셀 안에 `[수정]`으로 표시했습니다.

## 분석 범위와 용어

- 입력: Chapter 05 결과 `data/processed/*_clean.csv` 4개 (강사 저장소에 커밋된 파일과 동일한지 MD5와 git으로 확인)
- 분석 기간: 주문일 2025-07-01 ~ 2026-08-30. Chapter 05에서 확인한 유효 범위입니다. 주문일 결측 3건(72, 143, 214)과 작성일(2026-10-22) 이후 주문 2건(256: 2027-01-15, 287: 2028-12-31)은 제외합니다.
- 금액 분석 기준 데이터(이하 "기준 데이터"): `order_status == "completed"` 이고 분석 기간 안에 있는 주문 상세 437행, 주문 176건, 상품 주문 금액 138,528,000
  - 카테고리가 없는 16행(Chapter 05에서 제외된 상품 18·48·73·95를 참조, 4,015,000)은 버리지 않고 `(상품 정보 없음)`으로 따로 표시합니다. 그래서 카테고리별 합계와 월별 합계가 같은 범위가 됩니다.
  - category는 글자 사이 공백을 제거한 `category_clean`을 사용합니다(`뷰 티` → `뷰티` 등 3건).
- 금액 용어: `line_total = quantity × unit_price`는 상품 주문 금액입니다. 할인·쿠폰·배송비·세금·실제 결제액 정보가 없으므로 회계상 매출이라고 부르지 않습니다. 공식 셀의 컬럼명 `total_sales`, 보고서의 "매출"은 원문 그대로 두었습니다.
- 질문별로 필요한 테이블만 사용합니다. Q1~Q3에는 `customers`가 필요하지 않습니다. 고객 테이블은 공식 고객 셀과 과제 1·5에서만 연결합니다.

## 실습 가이드 STEP과 이 Notebook의 위치

| 가이드 STEP | 이 Notebook 위치 | 답안 양식 |
| --- | --- | --- |
| 0 제출용 Notebook 준비 | 공식 Notebook 복사본 | 제출 정보 |
| 1 전처리 데이터 준비 | 5번 + `[추가 검증] STEP 1` | — |
| 2 질문 → 지표 | 7번 + `[추가 검증] STEP 2` | 1 |
| 3 핵심 분포와 집계 | 8~13번 + `[추가 검증]` Q1·Q2·Q3 | 2 |
| 4 집계 총합 검증 | `[추가 검증] STEP 4` | 2 |
| 5 관찰·가설·추가 검증 | 사례 1~3 `[추가 검증]` | 3 |
| 6 LLM 질문 확장 | 20·21번 + `[추가 검증]` | 4 |
| 7 재현 가능한 EDA | 19번 + `[추가 검증] STEP 7` | 2 한계, 5 |
| 8 최종 해석 | 23번 뒤 | 5 |

## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
# [수정] 개인 저장소(llm-data-analysis-study/chapter06)에서 실행해도 강사 저장소 루트를 찾도록 경로 탐색만 바꿨습니다.
from pathlib import Path
import hashlib
import os
import subprocess
import sys

import pandas as pd
import numpy as np


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        for root in (candidate, candidate / 'llm-data-analysis-course'):
            if (root / 'requirements.txt').exists() and (root / 'scripts').exists():
                return root
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)

git = lambda *args: subprocess.run(['git', *args], cwd=PROJECT_ROOT, capture_output=True, text=True).stdout.strip()
print('Python 실행 파일:', sys.executable)
print('Python:', sys.version.split()[0], '/ pandas:', pd.__version__, '/ numpy:', np.__version__)
print('강사 저장소 clone:', git('remote', 'get-url', 'origin'), '| HEAD', git('log', '-1', '--format=%h %ci'))
print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)

Python 실행 파일: D:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
Python: 3.13.1 / pandas: 3.0.5 / numpy: 2.5.3


강사 저장소 clone: https://github.com/GilbertMoon/llm-data-analysis-course | HEAD 6105f42 2026-10-19 08:32:07 +0900
현재 실행 위치: D:\dev\llm-data-analysis-study\chapter06
프로젝트 루트: D:\dev\llm-data-analysis-course
전처리 데이터 폴더: D:\dev\llm-data-analysis-course\data\processed
보고서 폴더: D:\dev\llm-data-analysis-course\reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


In [3]:
# [추가 검증] STEP 1. 전처리 데이터가 Chapter 05 결과와 같은 파일인지 확인
FILE_NAMES = ['customers', 'products', 'orders', 'order_items']
md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
for n in FILE_NAMES:
    path = PROCESSED_DIR / f'{n}_clean.csv'
    print(f'{path.name:24s} {path.stat().st_size:>7,} bytes  md5={md5(path)}')
changed = git('status', '--short', '--', 'data/processed')
print('강사 저장소에 커밋된 Chapter 05 결과 대비 변경:', changed or '없음')
print('shape:', {n: d.shape for n, d in zip(FILE_NAMES, [customers, products, orders, order_items])})

customers_clean.csv        5,924 bytes  md5=3fbb57a59e99b53ced41e84623aa8d19
products_clean.csv         3,971 bytes  md5=57078017a0cb356794419d5ca8556888
orders_clean.csv          16,421 bytes  md5=8008b763340ed8ef3df48f8e0c746cf4
order_items_clean.csv     24,671 bytes  md5=bb3f3fa8216b38e2c40729ae3dc5b658
강사 저장소에 커밋된 Chapter 05 결과 대비 변경: 없음
shape: {'customers': (150, 6), 'products': (96, 4), 'orders': (300, 7), 'order_items': (752, 6)}


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [4]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (96, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (752, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [5]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


In [6]:
# [추가 검증] 분석 기간 확정: 날짜 결측과 작성일 이후 주문 제외
AS_OF = pd.Timestamp('2026-10-22')   # 작성일
VALID_START, VALID_END = pd.Timestamp('2025-07-01'), pd.Timestamp('2026-08-30')
nat = orders[orders['order_date'].isna()]
future = orders[orders['order_date'] > AS_OF]
print('order_date 결측 주문:', nat[['order_id', 'order_status']].to_dict('records'))
print('작성일 이후 주문:', [(int(r.order_id), r.order_date.strftime('%Y-%m-%d'), r.order_status) for r in future.itertuples()])
in_period = orders['order_date'].between(VALID_START, VALID_END)
print(f'분석 기간 {VALID_START.date()} ~ {VALID_END.date()} 주문: {int(in_period.sum())} / 전체 {len(orders)}')
print('기간 내 실제 첫·마지막 주문일:', orders.loc[in_period, 'order_date'].min().date(), '/', orders.loc[in_period, 'order_date'].max().date())

order_date 결측 주문: [{'order_id': 72, 'order_status': 'completed'}, {'order_id': 143, 'order_status': 'refunded'}, {'order_id': 214, 'order_status': 'completed'}]
작성일 이후 주문: [(256, '2027-01-15', 'completed'), (287, '2028-12-31', 'cancelled')]
분석 기간 2025-07-01 ~ 2026-08-30 주문: 295 / 전체 300
기간 내 실제 첫·마지막 주문일: 2025-07-01 / 2026-08-30


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [7]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


### STEP 2. 내가 정한 EDA 질문

공식 질문표(위 셀)는 출발점으로 두었습니다. 내 질문은 금액이 "얼마인가"가 아니라 "왜 다른가"를 나눠 볼 수 있도록 지표를 분해할 수 있는 형태로 정했습니다.

In [8]:
# [추가 검증] STEP 2. 내가 정한 EDA 질문 → 범위 → 데이터 → 지표 → 계산 → 검증
my_questions = pd.DataFrame([
    {'id': 'Q1', 'question': '카테고리별 completed 상품 주문 금액 차이는 판매 수량과 평균 판매 단가 중 무엇에서 오는가?',
     'scope': 'completed, 주문일 2025-07-01~2026-08-30',
     'data': 'order_items(quantity, unit_price, line_total) + orders(order_status, order_date) + products(category)',
     'metric': '상품 주문 금액, 판매 수량, 평균 판매 단가(금액/수량), 금액 비중',
     'validation': '카테고리 합계 = 기준 합계, 대량 주문 1건 제외 전후 순위 비교'},
    {'id': 'Q2', 'question': '월별 completed 상품 주문 금액 변화는 전체 주문 수, 완료율, 주문당 금액 중 무엇이 움직여서 생기는가?',
     'scope': '주문일 2025-07-01~2026-08-30 (완료율은 전체 상태, 금액은 completed)',
     'data': 'orders(order_date, order_status) + order_items(line_total)',
     'metric': '월별 상품 주문 금액, 전체 주문 수, 완료율, completed 주문당 금액',
     'validation': '월별 합계 = 기준 합계, 금액 = 전체 주문 수 × 완료율 × 주문당 금액 분해식 일치'},
    {'id': 'Q3', 'question': '취소·환불 비율은 결제수단과 카테고리에 따라 다른가?',
     'scope': '주문일 2025-07-01~2026-08-30, 전체 주문 상태',
     'data': 'orders(payment_method, order_status) + order_items + products(category)',
     'metric': '결제수단별·카테고리별 completed/cancelled/refunded 주문 비율',
     'validation': '상태별 건수 합 = 기간 내 주문 수, 카이제곱 검정으로 우연한 차이인지 확인'},
])
display(my_questions)

,id,question,scope,data,metric,validation
0,Q1,카테고리별 completed 상품 주문 금액 차이는 판매 수량과 평균 판매 단가 중 무엇에서 오는가?,"completed, 주문일 2025-07-01~2026-08-30","order_items(quantity, unit_price, line_total) + orders(order_status, order_d...","상품 주문 금액, 판매 수량, 평균 판매 단가(금액/수량), 금액 비중","카테고리 합계 = 기준 합계, 대량 주문 1건 제외 전후 순위 비교"
1,Q2,"월별 completed 상품 주문 금액 변화는 전체 주문 수, 완료율, 주문당 금액 중 무엇이 움직여서 생기는가?","주문일 2025-07-01~2026-08-30 (완료율은 전체 상태, 금액은 completed)","orders(order_date, order_status) + order_items(line_total)","월별 상품 주문 금액, 전체 주문 수, 완료율, completed 주문당 금액","월별 합계 = 기준 합계, 금액 = 전체 주문 수 × 완료율 × 주문당 금액 분해식 일치"
2,Q3,취소·환불 비율은 결제수단과 카테고리에 따라 다른가?,"주문일 2025-07-01~2026-08-30, 전체 주문 상태","orders(payment_method, order_status) + order_items + products(category)",결제수단별·카테고리별 completed/cancelled/refunded 주문 비율,"상태별 건수 합 = 기간 내 주문 수, 카이제곱 검정으로 우연한 차이인지 확인"


## 답안 1. 내가 정한 EDA 질문

| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| Q1. 카테고리별 completed 상품 주문 금액 차이는 판매 수량과 평균 판매 단가 중 무엇에서 오는가? | completed, 주문일 2025-07-01~2026-08-30 (기준 데이터 437행) | 상품 주문 금액, 판매 수량, 평균 판매 단가(금액 ÷ 수량), 금액 비중 | 카테고리 합계 = 기준 합계 138,528,000. 대량 주문 1건 제외 전후 순위 비교 |
| Q2. 월별 completed 상품 주문 금액 변화는 전체 주문 수, 완료율, 주문당 금액 중 무엇이 움직여서 생기는가? | 주문일 2025-07-01~2026-08-30. 주문 수·완료율은 전체 상태 295건, 금액은 기준 데이터 | 월별 상품 주문 금액, 전체 주문 수, 완료율, completed 주문당 금액 | 월별 합계 = 기준 합계. 금액 = 전체 주문 수 × 완료율 × 주문당 금액 분해식이 14개월 모두 일치 |
| Q3. 취소·환불 비율은 결제수단과 카테고리에 따라 다른가? | 주문일 2025-07-01~2026-08-30, 전체 주문 상태 295건 | 결제수단별 주문 상태 비율, 카테고리별 주문 상세 행·금액 기준 상태 비율 | 상태별 건수 합 = 295. 카이제곱 검정으로 우연한 차이일 가능성 확인 |

필요한 테이블과 연결 키:
- Q1: `order_items`(quantity, unit_price, line_total) → `orders`(order_status, order_date) [`order_id`, many-to-one] → `products`(category) [`product_id`, many-to-one]
- Q2: `order_items`(line_total) → `orders`(order_date, order_status) [`order_id`]. 상품 정보는 필요 없음
- Q3: `orders`(payment_method, order_status). 카테고리별 비교만 `order_items` → `products` 연결

### 왜 이 질문을 선택했는가?
- 현재 데이터로 답할 수 있는 질문이기 때문입니다. 세 질문에 필요한 컬럼(수량, 단가, 상태, 주문일, 결제수단, 카테고리)이 모두 있고, Chapter 05에서 결측·중복·관계를 확인해 둔 컬럼입니다. 반대로 "왜 고객이 환불했는가", "할인 효과" 같은 질문은 사유·할인 컬럼이 없어서 선택하지 않았습니다.
- 공식 질문 "카테고리별 매출은 어떻게 다른가?"는 순위만 알려 줍니다. 금액을 수량 × 단가, 주문 수 × 완료율 × 주문당 금액으로 분해하면 "어떤 요인이 차이를 만드는지"까지 계산으로 확인할 수 있습니다.
- Q3는 Q2와 연결됩니다. 월별 금액이 완료율에 따라 움직인다면, 다음 질문은 "완료되지 않는 주문이 어디에 몰려 있는가"이기 때문입니다.

## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [9]:
customers['age'].describe()


count    150.000000
mean      43.913333
std       17.802397
min       18.000000
25%       30.000000
50%       41.000000
75%       56.750000
max      150.000000
Name: age, dtype: float64

In [10]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [11]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,M,80,53.33
1,F,70,46.67


In [12]:
# [추가 검증] 고객 분포를 읽을 때 Chapter 05 전처리 흔적 확인
print('age == 41 고객:', int((customers['age'] == 41).sum()), '명 (그중 6명은 Chapter 05에서 중앙값 대체)')
print('age > 100:', customers.loc[customers['age'] > 100, ['customer_id', 'age']].to_dict('records'))
print('city Unknown:', int((customers['city'] == 'Unknown').sum()), '명 / 주문이 한 건도 없는 고객:',
      int((~customers['customer_id'].isin(orders['customer_id'])).sum()), '명')

age == 41 고객: 11 명 (그중 6명은 Chapter 05에서 중앙값 대체)
age > 100: [{'customer_id': 33, 'age': 150.0}]
city Unknown: 2 명 / 주문이 한 건도 없는 고객: 24 명


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [13]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,패션,16,16.67
1,뷰티,16,16.67
2,식품,16,16.67
3,도서,13,13.54
4,전자기기,13,13.54
5,스포츠,10,10.42
6,생활용품,9,9.38
7,전자 기기,1,1.04
8,생활 용품,1,1.04
9,뷰 티,1,1.04


In [14]:
products['price'].describe()


count    9.600000e+01
mean     2.048958e+05
std      1.011848e+06
min      5.000000e+03
25%      5.275000e+04
50%      9.450000e+04
75%      1.532500e+05
max      9.999999e+06
Name: price, dtype: float64

In [15]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
2,뷰티,16,710937.0,18000.0,9999999.0
3,생활 용품,1,124000.0,124000.0,124000.0
6,식품,16,120812.0,11000.0,195000.0
8,전자기기,13,120231.0,38000.0,178000.0
9,패션,16,115125.0,28000.0,197000.0
0,도서,13,103692.0,34000.0,182000.0
5,스포츠,10,78400.0,5000.0,183000.0
4,생활용품,9,69556.0,29000.0,147000.0
1,뷰 티,1,54000.0,54000.0,54000.0
7,전자 기기,1,21000.0,21000.0,21000.0


In [16]:
# [추가 검증] 공식 상품 집계의 함정: 표기 변형 카테고리와 극단 가격
print('공식 표의 category 수:', products['category'].nunique(), '→ 글자 사이 공백 제거 시', products['category'].str.replace(' ', '').nunique())
print('변형 표기:', sorted(c for c in products['category'].unique() if ' ' in c))
print('price 최대:', products.loc[products['price'].idxmax(), ['product_id', 'category', 'price']].to_dict())
beauty = products[products['category'] == '뷰티']['price']
print(f'뷰티 평균 가격: 상품 99 포함 {beauty.mean():,.0f} / 제외 {beauty[beauty < 1_000_000].mean():,.0f}')

# 이후 분석에서는 category 공백을 제거한 값을 사용합니다 (원본 컬럼은 유지).
products['category_clean'] = products['category'].str.replace(' ', '', regex=False)

공식 표의 category 수: 10 → 글자 사이 공백 제거 시 7
변형 표기: ['뷰 티', '생활 용품', '전자 기기']
price 최대: {'product_id': 99, 'category': '뷰티', 'price': 9999999.0}
뷰티 평균 가격: 상품 99 포함 710,937 / 제외 91,667


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [17]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,180,60.0
1,cancelled,63,21.0
2,refunded,57,19.0


In [18]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,90,30.00
1,card,72,24.00
2,bank_transfer,70,23.33
3,naver_pay,68,22.67


In [19]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00
주문 월 개수: 16


In [20]:
# [추가 검증] 공식 셀의 기간·월 수는 분석 기간 밖 주문을 포함
print('공식 주문 월 개수 14 + 2027-01 + 2028-12 = ', orders['order_month'].nunique(), '| order_month 결측', int(orders['order_month'].isna().sum()))
print('분석 기간 내 주문 월:', orders.loc[in_period, 'order_month'].nunique(), '개')
status_in_period = orders.loc[in_period, 'order_status'].value_counts()
print('분석 기간 내 주문 상태:', status_in_period.to_dict(), '합계', int(status_in_period.sum()))

공식 주문 월 개수 14 + 2027-01 + 2028-12 =  16 | order_month 결측 3
분석 기간 내 주문 월: 14 개
분석 기간 내 주문 상태: {'completed': 177, 'cancelled': 62, 'refunded': 56} 합계 295


## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [21]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [22]:
sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', sales_items['product_name'].isna().sum())
print('카테고리 누락:', sales_items['category'].isna().sum())

sales_items.head()


병합 전 order_items: (752, 6)
병합 후 sales_items: (752, 10)
상품명 누락: 26
카테고리 누락: 26


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price,category_clean
0,1,1,39,2.0,172000.0,344000.0,뷰티 상품 039,뷰티,172000.0,뷰티
1,2,2,26,4.0,79000.0,316000.0,뷰티 상품 026,뷰티,79000.0,뷰티
2,3,2,46,3.0,87000.0,261000.0,생활용품 상품 046,생활용품,87000.0,생활용품
3,4,2,80,1.0,187000.0,187000.0,패션 상품 080,패션,187000.0,패션
4,5,2,91,5.0,125000.0,625000.0,전자기기 상품 091,전자기기,125000.0,전자기기


In [23]:
# [추가 검증] 공식 merge에 validate·indicator를 붙여 다시 확인
check = order_items.merge(products, on='product_id', how='left', validate='many_to_one', indicator=True)
print('행 수', len(order_items), '→', len(check), '| _merge:', check['_merge'].value_counts().to_dict())
missing = check[check['_merge'] == 'left_only']
print('카테고리 없는 행의 product_id:', missing['product_id'].value_counts().to_dict())
print('카테고리 없는 행의 line_total 합계:', f"{missing['line_total'].sum():,.0f}")

행 수 752 → 752 | _merge: {'both': 726, 'left_only': 26, 'right_only': 0}
카테고리 없는 행의 product_id: {48: 9, 73: 6, 18: 5, 95: 4, 9991: 1, 9992: 1}
카테고리 없는 행의 line_total 합계: 6,224,000


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [24]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
9,패션,545.0,64890000.0,26.39
6,식품,394.0,47044000.0,19.13
8,전자기기,303.0,37542000.0,15.27
2,뷰티,282.0,28614000.0,11.64
0,도서,277.0,28246000.0,11.49
5,스포츠,254.0,20651000.0,8.40
4,생활용품,213.0,14303000.0,5.82
3,생활 용품,27.0,3348000.0,1.36
1,뷰 티,19.0,1026000.0,0.42
7,전자 기기,12.0,252000.0,0.10


In [25]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,패션,545.0,64890000.0,26.39,115125.0
1,식품,394.0,47044000.0,19.13,120812.0
2,전자기기,303.0,37542000.0,15.27,120231.0
3,뷰티,282.0,28614000.0,11.64,710937.0
4,도서,277.0,28246000.0,11.49,103692.0
5,스포츠,254.0,20651000.0,8.40,78400.0
6,생활용품,213.0,14303000.0,5.82,69556.0
7,생활 용품,27.0,3348000.0,1.36,124000.0
8,뷰 티,19.0,1026000.0,0.42,54000.0
9,전자 기기,12.0,252000.0,0.10,21000.0


In [26]:
# [추가 검증] 공식 category_sales는 주문 상태를 거르지 않은 값입니다.
status_map = orders.set_index('order_id')['order_status']
print('공식 category_sales 합계:', f"{category_sales['total_sales'].sum():,.0f}", '(cancelled·refunded 포함, 카테고리 없는 행 제외)')
print('그중 completed 비중:', f"{order_items.loc[order_items['order_id'].map(status_map).eq('completed'), 'line_total'].sum() / order_items['line_total'].sum() * 100:.2f}%")
print('공식 표의 category 행 수:', len(category_sales), '(표기 변형 포함)')

공식 category_sales 합계: 245,916,000 (cancelled·refunded 포함, 카테고리 없는 행 제외)
그중 completed 비중: 56.06%
공식 표의 category 행 수: 10 (표기 변형 포함)


### 기준 데이터 만들기 (completed + 분석 기간)

공식 셀의 카테고리·월별 집계는 주문 상태를 거르지 않아, 취소·환불된 주문 금액까지 들어 있습니다(위 셀: completed 비중 56.06%). 금액 분석은 가이드 STEP 3 기준대로 completed 범위로 다시 계산합니다. 단계마다 행·주문·금액이 얼마나 빠지는지 남깁니다.

In [27]:
# [추가 검증] 분석 기준 데이터: completed + 분석 기간. 카테고리가 없는 행은 버리지 않고 별도 라벨로 남김
items = order_items.merge(orders[['order_id', 'customer_id', 'order_date', 'order_status', 'payment_method']],
                          on='order_id', how='left', validate='many_to_one', indicator='_order')
items = items.merge(products[['product_id', 'category_clean']], on='product_id', how='left',
                    validate='many_to_one', indicator='_product')
items['category_clean'] = items['category_clean'].fillna('(상품 정보 없음)')
items['in_period'] = items['order_date'].between(VALID_START, VALID_END)

base = items[(items['order_status'] == 'completed') & items['in_period']].copy()
base['order_month'] = base['order_date'].dt.to_period('M').astype(str)
BASE_TOTAL = base['line_total'].sum()

steps = pd.DataFrame([
    {'단계': '전처리 후 order_items 전체', '행': len(items), '주문': items['order_id'].nunique(), '금액': items['line_total'].sum()},
    {'단계': 'orders와 연결 안 됨 (order_id 9991·9992)', '행': int((items['_order'] == 'left_only').sum()), '주문': None,
     '금액': items.loc[items['_order'] == 'left_only', 'line_total'].sum()},
    {'단계': 'completed', '행': int((items['order_status'] == 'completed').sum()),
     '주문': items.loc[items['order_status'] == 'completed', 'order_id'].nunique(),
     '금액': items.loc[items['order_status'] == 'completed', 'line_total'].sum()},
    {'단계': 'completed + 분석 기간 (기준)', '행': len(base), '주문': base['order_id'].nunique(), '금액': BASE_TOTAL},
])
display(steps)
completed_orders_in_period = orders[(orders['order_status'] == 'completed') & in_period]
print('기간 내 completed 주문(orders 기준):', len(completed_orders_in_period), '/ 상세 행이 있는 completed 주문:', base['order_id'].nunique())
print('상세 행이 없는 completed 주문:', sorted(set(completed_orders_in_period['order_id']) - set(base['order_id'])))
print('기준 데이터 중 카테고리 없는 행:', int((base['category_clean'] == '(상품 정보 없음)').sum()), '행')

,단계,행,주문,금액
0,전처리 후 order_items 전체,752,301.0,252140000.0
1,orders와 연결 안 됨 (order_id 9991·9992),2,NaN,922000.0
2,completed,446,179.0,141340000.0
3,completed + 분석 기간 (기준),437,176.0,138528000.0


기간 내 completed 주문(orders 기준): 177 / 상세 행이 있는 completed 주문: 176
상세 행이 없는 completed 주문: [175]
기준 데이터 중 카테고리 없는 행: 16 행


In [28]:
# [추가 검증] Q1. 카테고리별 completed 상품 주문 금액 = 판매 수량 × 평균 판매 단가
catalog = products.groupby('category_clean')['product_id'].nunique()
q1 = (base.groupby('category_clean')
      .agg(amount=('line_total', 'sum'), quantity=('quantity', 'sum'), item_rows=('line_total', 'size'),
           orders=('order_id', 'nunique'), sold_products=('product_id', 'nunique'))
      .sort_values('amount', ascending=False))
q1['avg_unit_price'] = (q1['amount'] / q1['quantity']).round(0)
q1['amount_share(%)'] = (q1['amount'] / q1['amount'].sum() * 100).round(2)
q1['catalog_products'] = catalog
display(q1)
print('카테고리 합계:', f"{q1['amount'].sum():,.0f}", '/ 기준 합계:', f'{BASE_TOTAL:,.0f}', '/ 일치:', q1['amount'].sum() == BASE_TOTAL)

real = q1.drop(index='(상품 정보 없음)')
print('\n금액 순위와 상관 (카테고리 7개, 참고용):')
print('  금액 vs 판매 수량 상관계수:', round(real['amount'].corr(real['quantity']), 3))
print('  금액 vs 평균 판매 단가 상관계수:', round(real['amount'].corr(real['avg_unit_price']), 3))

,amount,quantity,item_rows,orders,sold_products,avg_unit_price,amount_share(%),catalog_products
category_clean,,,,,,,,
패션,32940000.0,290.0,76,68,15,113586.0,23.78,16.0
식품,24345000.0,218.0,76,60,16,111674.0,17.57,16.0
전자기기,20445000.0,172.0,57,53,14,118866.0,14.76,14.0
뷰티,17542000.0,178.0,64,54,17,98551.0,12.66,17.0
도서,16895000.0,168.0,53,47,13,100565.0,12.20,13.0
스포츠,13237000.0,165.0,53,47,10,80224.0,9.56,10.0
생활용품,9109000.0,125.0,42,42,10,72872.0,6.58,10.0
(상품 정보 없음),4015000.0,34.0,16,16,5,118088.0,2.90,NaN


카테고리 합계: 138,528,000 / 기준 합계: 138,528,000 / 일치: True

금액 순위와 상관 (카테고리 7개, 참고용):
  금액 vs 판매 수량 상관계수: 0.968
  금액 vs 평균 판매 단가 상관계수: 0.82


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [29]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()


병합 후 order_sales:

 (752, 12)
order_date 누락: 11


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,39,2.0,172000.0,344000.0,144.0,2026-07-12,naver_pay,completed,2026-07,Sunday
1,2,2,26,4.0,79000.0,316000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
2,3,2,46,3.0,87000.0,261000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
3,4,2,80,1.0,187000.0,187000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
4,5,2,91,5.0,125000.0,625000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday


In [30]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-07,13729000.0,19,722579.0
1,2025-08,32639000.0,25,1305560.0
2,2025-09,13058000.0,12,1088167.0
3,2025-10,16680000.0,22,758182.0
4,2025-11,15080000.0,23,655652.0
5,2025-12,16152000.0,23,702261.0
6,2026-01,19053000.0,26,732808.0
7,2026-02,14207000.0,19,747737.0
8,2026-03,13841000.0,22,629136.0
9,2026-04,21007000.0,26,807962.0


In [31]:
# [추가 검증] 공식 monthly_sales 범위 확인
print('공식 monthly_sales 행:', len(monthly_sales), '| 기간 밖 월:', [m for m in monthly_sales['order_month'] if m < '2025-07' or m > '2026-08'])
print('공식 합계:', f"{monthly_sales['total_sales'].sum():,.0f}", '| order_items 전체 합계:', f"{order_items['line_total'].sum():,.0f}")
print('→ 차이', f"{order_items['line_total'].sum() - monthly_sales['total_sales'].sum():,.0f}",
      '= order_month가 결측인 행(날짜 결측 주문 + orders에 없는 주문)이 groupby에서 조용히 빠진 금액')

공식 monthly_sales 행: 16 | 기간 밖 월: ['2027-01', '2028-12']
공식 합계: 248,630,000 | order_items 전체 합계: 252,140,000
→ 차이 3,510,000 = order_month가 결측인 행(날짜 결측 주문 + orders에 없는 주문)이 groupby에서 조용히 빠진 금액


In [32]:
# [추가 검증] Q2. 월별 completed 상품 주문 금액 = 전체 주문 수 × 완료율 × completed 주문당 금액
period_orders = orders[in_period].copy()
month_all = period_orders.groupby('order_month').agg(all_orders=('order_id', 'nunique'),
                                                      completed_orders=('order_status', lambda s: int((s == 'completed').sum())))
month_amt = base.groupby('order_month').agg(amount=('line_total', 'sum'), orders_with_items=('order_id', 'nunique'),
                                            quantity=('quantity', 'sum'))
q2 = month_all.join(month_amt)
q2['completion_rate(%)'] = (q2['completed_orders'] / q2['all_orders'] * 100).round(1)
q2['amount_per_completed_order'] = (q2['amount'] / q2['completed_orders']).round(0)
q2['check'] = (q2['all_orders'] * (q2['completed_orders'] / q2['all_orders']) * (q2['amount'] / q2['completed_orders'])).round(0) == q2['amount']
display(q2)
print('월별 합계:', f"{q2['amount'].sum():,.0f}", '/ 기준 합계:', f'{BASE_TOTAL:,.0f}', '/ 일치:', q2['amount'].sum() == BASE_TOTAL)
print('분해식 일치 월:', int(q2['check'].sum()), '/', len(q2))
print('월별 금액과의 상관계수 — 전체 주문 수:', round(q2['amount'].corr(q2['all_orders']), 3),
      '/ 완료율:', round(q2['amount'].corr(q2['completion_rate(%)']), 3),
      '/ completed 주문 수:', round(q2['amount'].corr(q2['completed_orders']), 3),
      '/ 주문당 금액:', round(q2['amount'].corr(q2['amount_per_completed_order']), 3))

,all_orders,completed_orders,amount,orders_with_items,quantity,completion_rate(%),amount_per_completed_order,check
order_month,,,,,,,,
2025-07,19,11,9561000.0,11,93.0,57.9,869182.0,True
2025-08,26,18,9362000.0,17,106.0,69.2,520111.0,True
2025-09,12,4,5299000.0,4,50.0,33.3,1324750.0,True
2025-10,22,17,12629000.0,17,120.0,77.3,742882.0,True
2025-11,23,16,9583000.0,16,100.0,69.6,598938.0,True
2025-12,23,12,8581000.0,12,85.0,52.2,715083.0,True
2026-01,26,9,8135000.0,9,77.0,34.6,903889.0,True
2026-02,19,12,7689000.0,12,73.0,63.2,640750.0,True
2026-03,22,13,7247000.0,13,81.0,59.1,557462.0,True


월별 합계: 138,528,000 / 기준 합계: 138,528,000 / 일치: True
분해식 일치 월: 14 / 14
월별 금액과의 상관계수 — 전체 주문 수: 0.295 / 완료율: 0.599 / completed 주문 수: 0.576 / 주문당 금액: 0.104


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [33]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

# [수정] 공식 셀은 name이 있으면 이름으로도 묶어 출력합니다. 분석에 필요하지 않은 개인정보라
#        customer_id와 city만 사용합니다. (저장되는 ch06_customer_sales.csv에도 name이 들어가지 않습니다.)
group_columns = ['customer_id', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)

,customer_id,city,order_count,total_sales,avg_order_value
15,18.0,대전,2,16666000.0,8333000.0
2,3.0,서울,7,8909000.0,1272714.0
17,20.0,성남,6,6200000.0,1033333.0
45,52.0,울산,5,5388000.0,1077600.0
11,14.0,대구,4,5204000.0,1301000.0
19,22.0,부산,4,5029000.0,1257250.0
48,55.0,부산,6,4850000.0,808333.0
74,90.0,수원,5,4847000.0,969400.0
12,15.0,광주,5,4281000.0,856200.0
77,93.0,울산,4,4001000.0,1000250.0


In [34]:
# [추가 검증] 공식 고객 집계의 범위 확인 (주문 상태 미구분, 고객 없는 주문 9991·9992 제외됨)
print('공식 customer_sales 고객 수:', len(customer_sales), '| 합계:', f"{customer_sales['total_sales'].sum():,.0f}")
print('customer_id가 customers에 없는 주문:', sorted(int(x) for x in order_sales.loc[~order_sales['customer_id'].isin(customers['customer_id']), 'order_id'].unique()))
top = customer_sales.iloc[0]
top_status = order_sales[order_sales['customer_id'] == top['customer_id']].groupby(['order_id', 'order_status'])['line_total'].sum()
print(f"공식 1위 customer_id {int(top['customer_id'])}의 주문:", {f'{k[0]}({k[1]})': v for k, v in top_status.items()})
aug = order_sales[order_sales['order_month'] == '2025-08'].groupby('order_status')['line_total'].sum()
print('공식 월별 최고 2025-08의 상태별 금액:', aug.to_dict())

공식 customer_sales 고객 수: 126 | 합계: 249,159,000
customer_id가 customers에 없는 주문: [126, 278, 9991, 9992]
공식 1위 customer_id 18의 주문: {'131(cancelled)': 1166000.0, '295(refunded)': 15500000.0}
공식 월별 최고 2025-08의 상태별 금액: {'cancelled': 3682000.0, 'completed': 9362000.0, 'refunded': 19595000.0}


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [35]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


In [36]:
# [추가 검증] Q3. 결제수단별 주문 상태 비율 + 카이제곱 검정
from scipy.stats import chi2_contingency

pay_status = pd.crosstab(period_orders['payment_method'], period_orders['order_status'])
pay_rate = (pay_status.div(pay_status.sum(axis=1), axis=0) * 100).round(1).add_suffix('(%)')
q3_pay = pay_status.join(pay_rate)
q3_pay['orders'] = pay_status.sum(axis=1)
display(q3_pay)
print('상태별 건수 합:', int(pay_status.values.sum()), '/ 기간 내 주문:', len(period_orders))
chi2, pval, dof, _ = chi2_contingency(pay_status)
print(f'결제수단 × 주문 상태 카이제곱: chi2={chi2:.2f}, dof={dof}, p={pval:.3f}')

# 카테고리별: 주문 상세 행 기준 (한 주문에 여러 카테고리가 있을 수 있어 행 단위로 봄)
period_items = items[items['in_period'] & (items['_order'] == 'both')]
cat_status = pd.crosstab(period_items['category_clean'], period_items['order_status'])
cat_amt_rate = (pd.crosstab(period_items['category_clean'], period_items['order_status'],
                            values=period_items['line_total'], aggfunc='sum', normalize='index') * 100).round(1)
display(cat_status.join(cat_amt_rate.add_suffix(' 금액(%)')))
chi2c, pvalc, dofc, _ = chi2_contingency(cat_status.drop(index='(상품 정보 없음)'))
print(f'카테고리 × 주문 상태(행 기준) 카이제곱: chi2={chi2c:.2f}, dof={dofc}, p={pvalc:.3f}')

order_status,cancelled,completed,refunded,cancelled(%),completed(%),refunded(%),orders
payment_method,,,,,,,
bank_transfer,10,48,11,14.5,69.6,15.9,69
card,11,40,20,15.5,56.3,28.2,71
kakao_pay,25,49,15,28.1,55.1,16.9,89
naver_pay,16,40,10,24.2,60.6,15.2,66


상태별 건수 합: 295 / 기간 내 주문: 295
결제수단 × 주문 상태 카이제곱: chi2=10.67, dof=6, p=0.099


order_status,cancelled,completed,refunded,cancelled 금액(%),completed 금액(%),refunded 금액(%)
category_clean,,,,,,
(상품 정보 없음),3,16,7,7.5,64.5,28.0
도서,17,53,22,19.7,61.5,18.9
뷰티,21,64,18,21.4,60.1,18.5
생활용품,19,42,18,25.3,52.4,22.3
스포츠,16,53,12,20.9,64.1,15.0
식품,32,76,23,26.5,54.3,19.2
전자기기,21,57,19,25.2,57.2,17.6
패션,27,76,22,16.0,51.1,32.9


카테고리 × 주문 상태(행 기준) 카이제곱: chi2=5.53, dof=12, p=0.938


### STEP 4. 집계 총합 검증

같은 기준 데이터에서 나온 집계라면 어느 기준으로 나누든 합계가 같아야 합니다. 카테고리, 월, 결제수단, 요일, 주문, 월×카테고리, 고객 연결 여부의 8가지 합계를 기준 합계와 비교했습니다. 하나라도 다르면 `assert`로 멈추게 했습니다.

In [37]:
# [추가 검증] STEP 4. 집계 총합 검증 — 같은 기준 데이터(base)에서 나온 모든 집계가 같은 합계인지
order_amount = base.groupby('order_id')['line_total'].sum()
cust_linked = base['customer_id'].isin(customers['customer_id'])
total_check = pd.DataFrame([
    ('기준: completed + 분석 기간 주문 상세 line_total', BASE_TOTAL),
    ('Q1 카테고리별 합계 (상품 정보 없음 포함)', q1['amount'].sum()),
    ('Q2 월별 합계', q2['amount'].sum()),
    ('결제수단별 합계', base.groupby('payment_method')['line_total'].sum().sum()),
    ('요일별 합계', base.groupby(base['order_date'].dt.day_name())['line_total'].sum().sum()),
    ('주문별 합계', order_amount.sum()),
    ('월×카테고리 피벗 합계', base.pivot_table(index='order_month', columns='category_clean', values='line_total', aggfunc='sum').sum().sum()),
    ('고객 연결 + 고객 미연결', base.loc[cust_linked, 'line_total'].sum() + base.loc[~cust_linked, 'line_total'].sum()),
], columns=['집계', '합계'])
total_check['차이'] = total_check['합계'] - BASE_TOTAL
total_check['결과'] = np.where(total_check['차이'] == 0, 'PASS', 'FAIL')
display(total_check)
print('주문 수 검증: Q2 completed 주문 수 합', int(q2['completed_orders'].sum()), '= 상세 있는 주문', base['order_id'].nunique(), '+ 상세 없는 주문 1 (175)')
print('수량 검증: Q1 수량 합', int(q1['quantity'].sum()), '= 기준 수량', int(base['quantity'].sum()))
assert (total_check['결과'] == 'PASS').all()

,집계,합계,차이,결과
0,기준: completed + 분석 기간 주문 상세 line_total,138528000.0,0.0,PASS
1,Q1 카테고리별 합계 (상품 정보 없음 포함),138528000.0,0.0,PASS
2,Q2 월별 합계,138528000.0,0.0,PASS
3,결제수단별 합계,138528000.0,0.0,PASS
4,요일별 합계,138528000.0,0.0,PASS
5,주문별 합계,138528000.0,0.0,PASS
6,월×카테고리 피벗 합계,138528000.0,0.0,PASS
7,고객 연결 + 고객 미연결,138528000.0,0.0,PASS


주문 수 검증: Q2 completed 주문 수 합 177 = 상세 있는 주문 176 + 상세 없는 주문 1 (175)
수량 검증: Q1 수량 합 1350 = 기준 수량 1350


## 답안 2. 핵심 집계 결과

- completed 원본 합계: 138,528,000 (completed + 주문일 2025-07-01~2026-08-30, 주문 상세 437행, 주문 176건, 수량 1,350)
- 카테고리별 합계: 패션 32,940,000 (23.78%) > 식품 24,345,000 (17.57%) > 전자기기 20,445,000 (14.76%) > 뷰티 17,542,000 (12.66%) > 도서 16,895,000 (12.20%) > 스포츠 13,237,000 (9.56%) > 생활용품 9,109,000 (6.58%) + 상품 정보 없음 4,015,000 (2.90%) = 138,528,000
- 월별 합계: 최고 2026-08 16,980,000 (completed 15건), 최저 2025-09 5,299,000 (completed 4건). 14개월 합계 138,528,000
- 총합 일치 여부: 8개 집계 모두 PASS (차이 0). 주문 수(177 = 상세 있는 176 + 상세 없는 1)와 수량(1,350)도 일치

![핵심 EDA 결과](images/step02_eda.png)

- Evidence: `[추가 검증]` 기준 데이터, Q1, Q2, Q3, STEP 4 셀 Output (이미지: `images/step02_eda.png`, `images/step03_total_check.png`)

### 결과 관찰
- Q1: 카테고리 금액은 판매 수량과의 상관계수가 0.968, 평균 판매 단가와는 0.82였습니다(카테고리 7개). 패션은 판매 수량 290으로 가장 많고, 평균 판매 단가는 113,586으로 7개 중 3위입니다.
- Q2: 월별 금액과의 상관계수는 완료율 0.599, completed 주문 수 0.576, 전체 주문 수 0.295, 주문당 금액 0.104였습니다(14개월). 완료율은 2025-09 33.3%, 2026-01 34.6%에서 2025-10 77.3%까지 차이가 큽니다.
- Q3: 결제수단별 completed 비율은 bank_transfer 69.6%, naver_pay 60.6%, card 56.3%, kakao_pay 55.1%입니다. card의 환불 비율이 28.2%로 다른 수단(15.2~16.9%)보다 높고, kakao_pay는 취소 비율이 28.1%입니다. 결제수단 × 상태 카이제곱 p=0.099, 카테고리 × 상태 p=0.938입니다.
- 공식 셀(상태 미구분) 결과와 비교하면 결론이 달라졌습니다.
  - 공식 월별 최고는 2025-08(32,639,000)인데, 그중 19,595,000(60%)이 refunded입니다. completed 기준 2025-08은 9,362,000으로 평균 수준입니다.
  - 공식 고객별 1위(customer_id 18, 16,666,000)는 cancelled 1건과 refunded 1건(15,500,000)뿐이고 completed 금액은 0입니다.

### 나의 해석과 판단
- 가장 의미 있다고 본 결과는 "범위 정의가 결론을 바꾼다"는 점입니다. 공식 표만 보면 2025-08이 최고 달이고 customer 18이 최우수 고객이지만, 둘 다 환불된 대량 주문(295, 수량 100) 한 건이 만든 숫자였습니다. 금액 분석에서 completed 필터는 선택 사항이 아니라 필수라고 판단했습니다.
- Q1에서는 카테고리 차이를 수량이 더 잘 설명했습니다. 다만 카테고리가 7개뿐이라 상관계수는 방향을 보는 참고값입니다.
- Q2에서는 월별 금액을 주문당 금액보다 완료율과 completed 주문 수가 더 잘 설명했습니다. "주문이 적어서 금액이 낮다"보다 "주문은 들어왔는데 완료되지 않아 금액이 낮다"는 달(2025-09, 2026-01)이 있었습니다.
- Q3의 결제수단 차이는 전체 검정에서 p=0.099로 5% 기준을 넘었습니다. "결제수단에 따라 다르다"고 결론 내리기에는 근거가 부족하다고 판단했습니다. 카테고리별 상태 비율은 차이가 거의 없습니다(p=0.938).

### 업무·분석적 의미
- 카테고리 운영 관점에서는 단가보다 판매 수량이 금액 차이를 만들고 있어, 수량이 많은 패션·식품을 먼저 자세히 볼 가치가 있습니다.
- 월별 성과를 볼 때는 상품 주문 금액 하나만 보지 말고 완료율을 같이 보고해야 합니다. 완료율이 낮은 달은 수요가 아니라 취소·환불 처리 쪽을 확인해야 합니다.
- 공식 스크립트 결과(`reports/ch06_*.csv`)를 그대로 Chapter 07 시각화에 쓰면 환불 주문이 만든 2025-08 최고점을 그래프로 그리게 됩니다. 시각화 전에 completed 기준 표로 바꿔야 합니다.

### 한계와 추가 확인 사항
- 월 14개, 카테고리 7개, 결제수단 4개로 표본이 작습니다. 상관계수와 검정 결과는 경향을 보는 수준입니다.
- `line_total`은 상품 주문 금액이라 실제 결제액·할인·배송비는 반영하지 못합니다.
- 카테고리가 없는 16행(4,015,000)은 어느 카테고리인지 알 수 없어 카테고리 순위에 영향을 줄 수 있습니다. 이 금액이 모두 한 카테고리였다고 가정해도 1위(패션)와 2위(식품)의 차이 8,595,000보다 작아 1위는 바뀌지 않습니다.

## 답안 3. 관찰 → 가설 → 추가 검증

가이드 STEP 5대로 관찰(데이터에서 직접 확인한 사실), 가설(가능성), 추가 검증(가설을 확인하기 위해 계산할 것)을 나눴습니다. 추가 검증 중 지금 데이터로 바로 계산할 수 있는 것은 아래 셀에서 실행했고, 그 결과도 함께 적었습니다.

In [38]:
# [추가 검증] 사례 1: 패션 1위가 대량 주문 1건 때문인가?
big = base.nlargest(3, 'line_total')[['order_id', 'order_date', 'category_clean', 'quantity', 'unit_price', 'line_total']]
display(big)
no_big = base[base['quantity'] <= 10]
rank_with = q1.drop(index='(상품 정보 없음)')['amount'].rank(ascending=False).astype(int)
q1_no_big = no_big.groupby('category_clean').agg(amount=('line_total', 'sum'), quantity=('quantity', 'sum'))
q1_no_big = q1_no_big.drop(index='(상품 정보 없음)')
q1_no_big['avg_unit_price'] = (q1_no_big['amount'] / q1_no_big['quantity']).round(0)
q1_no_big['rank_without'] = q1_no_big['amount'].rank(ascending=False).astype(int)
q1_no_big['rank_with'] = rank_with
display(q1_no_big.sort_values('amount', ascending=False))
fashion_qty = base[base['category_clean'] == '패션']['quantity']
print('패션 판매 수량', int(fashion_qty.sum()), '중 주문 290 한 행', int(fashion_qty.max()), f'({fashion_qty.max() / fashion_qty.sum() * 100:.1f}%)')
print('패션 상품 16개 중 completed 판매가 있는 상품:', base.loc[base['category_clean'] == '패션', 'product_id'].nunique())

,order_id,order_date,category_clean,quantity,unit_price,line_total
726,290,2026-04-24,패션,50.0,96000.0,4800000.0
314,129,2026-07-01,패션,5.0,197000.0,985000.0
19,9,2026-05-12,뷰티,5.0,193000.0,965000.0


,amount,quantity,avg_unit_price,rank_without,rank_with
category_clean,,,,,
패션,28140000.0,240.0,117250.0,1,1
식품,24345000.0,218.0,111674.0,2,2
전자기기,20445000.0,172.0,118866.0,3,3
뷰티,17542000.0,178.0,98551.0,4,4
도서,16895000.0,168.0,100565.0,5,5
스포츠,13237000.0,165.0,80224.0,6,6
생활용품,9109000.0,125.0,72872.0,7,7


패션 판매 수량 290 중 주문 290 한 행 50 (17.2%)
패션 상품 16개 중 completed 판매가 있는 상품: 15


In [39]:
# [추가 검증] 사례 2: 2025-09 최저, 2026-08 최고는 무엇이 움직였나? (전체 평균 대비)
avg = pd.Series({'all_orders': q2['all_orders'].mean(), 'completion_rate(%)': q2['completion_rate(%)'].mean(),
                 'amount_per_completed_order': q2['amount_per_completed_order'].mean(), 'amount': q2['amount'].mean()})
cmp = q2.loc[['2025-09', '2026-01', '2026-08'], ['all_orders', 'completion_rate(%)', 'amount_per_completed_order', 'amount']].T
cmp['월 평균'] = avg.round(1)
display(cmp)
for m in ['2025-09', '2026-01', '2026-08']:
    st = period_orders[period_orders['order_month'] == m]['order_status'].value_counts().to_dict()
    days = period_orders[period_orders['order_month'] == m]['order_date'].dt.date.nunique()
    print(m, '상태:', st, '| 주문이 있는 날:', days, '일')
aug_cat = base[base['order_month'] == '2026-08'].groupby('category_clean')['line_total'].sum().sort_values(ascending=False)
print('2026-08 카테고리별:', aug_cat.to_dict())
print('2026-08 주문당 금액 상위 3건:', base[base['order_month'] == '2026-08'].groupby('order_id')['line_total'].sum().nlargest(3).to_dict())

order_month,2025-09,2026-01,2026-08,월 평균
all_orders,12.0,26.0,21.0,21.1
completion_rate(%),33.3,34.6,71.4,59.6
amount_per_completed_order,1324750.0,903889.0,1132000.0,831016.8
amount,5299000.0,8135000.0,16980000.0,9894857.1


2025-09 상태: {'refunded': 5, 'completed': 4, 'cancelled': 3} | 주문이 있는 날: 11 일
2026-01 상태: {'completed': 9, 'refunded': 9, 'cancelled': 8} | 주문이 있는 날: 17 일
2026-08 상태: {'completed': 15, 'cancelled': 4, 'refunded': 2} | 주문이 있는 날: 18 일
2026-08 카테고리별: {'패션': 4247000.0, '전자기기': 3212000.0, '도서': 2929000.0, '식품': 2661000.0, '뷰티': 1648000.0, '생활용품': 992000.0, '스포츠': 979000.0, '(상품 정보 없음)': 312000.0}
2026-08 주문당 금액 상위 3건: {50: 2088000.0, 108: 1891000.0, 225: 1878000.0}


In [40]:
# [추가 검증] 사례 3: 카드 결제의 환불 비율, 패션의 환불 금액 비율은 우연한 차이인가?
from scipy.stats import fisher_exact
card = pay_status.loc['card']
others = pay_status.drop(index='card').sum()
table = [[card['refunded'], card.sum() - card['refunded']], [others['refunded'], others.sum() - others['refunded']]]
odds, p_card = fisher_exact(table)
print(f"카드 환불 {card['refunded']}/{card.sum()} ({card['refunded'] / card.sum() * 100:.1f}%) vs 기타 {others['refunded']}/{others.sum()} "
      f"({others['refunded'] / others.sum() * 100:.1f}%) → Fisher p={p_card:.3f}")
fashion_ref = period_items[(period_items['category_clean'] == '패션') & (period_items['order_status'] == 'refunded')]
print('패션 환불 금액:', f"{fashion_ref['line_total'].sum():,.0f}", '| 그중 주문 295 이외 행:', f"{fashion_ref.loc[fashion_ref['quantity'] <= 10, 'line_total'].sum():,.0f}")
print('패션 환불 행 수:', len(fashion_ref), '/ 패션 전체 행:', int((period_items['category_clean'] == '패션').sum()))
big_ref = period_items[period_items['quantity'] > 10][['order_id', 'category_clean', 'quantity', 'line_total', 'order_status']]
print('수량 10 초과 행:', big_ref.to_dict('records'))

카드 환불 20/71 (28.2%) vs 기타 36/224 (16.1%) → Fisher p=0.036
패션 환불 금액: 21,220,000 | 그중 주문 295 이외 행: 5,720,000
패션 환불 행 수: 22 / 패션 전체 행: 125
수량 10 초과 행: [{'order_id': 290, 'category_clean': '패션', 'quantity': 50.0, 'line_total': 4800000.0, 'order_status': 'completed'}, {'order_id': 295, 'category_clean': '패션', 'quantity': 100.0, 'line_total': 15500000.0, 'order_status': 'refunded'}]


### 사례 1. 패션 1위
- 관찰: completed 상품 주문 금액 1위는 패션 32,940,000 (23.78%)입니다. 패션에는 단일 행 최대 금액인 주문 290(2026-04-24, 수량 50, 4,800,000)이 포함되어 있습니다.
- 가설: 패션 1위는 대량 주문 1건 때문일 수 있다.
- 추가 검증 (실행함): 수량 10 초과 행을 빼고 다시 계산했습니다. 패션은 28,140,000으로 여전히 1위이고, 7개 카테고리 순위가 하나도 바뀌지 않았습니다. 대량 주문은 패션 판매 수량의 17.2%를 차지하지만, 이것을 빼도 패션 수량(240)이 식품(218)보다 많습니다. 그래서 "대량 주문 1건 때문"이라는 가설은 이번 데이터에서 지지되지 않았습니다.
- 남은 추가 검증: 패션 수량이 많은 이유가 특정 상품에 몰린 것인지(상품별 수량 상위 5개 비중), 주문당 수량이 많은 것인지(주문당 평균 수량)를 다음에 계산합니다. 주문 290이 기업 대량 주문인지는 업무 확인이 필요합니다.

### 사례 2. 2025-09 최저, 2026-08 최고
- 관찰: 2025-09는 전체 주문 12건, 완료율 33.3%(completed 4, refunded 5, cancelled 3)로 상품 주문 금액이 5,299,000입니다. 2026-01도 전체 주문은 26건으로 평균(21.1건)보다 많은데 완료율이 34.6%여서 8,135,000에 그쳤습니다. 2026-08은 전체 21건, 완료율 71.4%, 주문당 금액 1,132,000으로 16,980,000입니다.
- 가설: 월별 금액 차이는 주문 유입보다 완료율(취소·환불)에서 더 크게 생긴다. 2026-08 최고는 특정 고액 주문이 아니라 여러 카테고리가 함께 늘어난 결과일 수 있다.
- 추가 검증 (실행함): 금액과의 상관계수는 완료율 0.599 > 전체 주문 수 0.295였습니다. 2026-01은 주문이 평균보다 많았는데도 완료율 때문에 금액이 낮았습니다. 2026-08은 7개 카테고리 모두 금액이 있고, 주문당 금액 상위 3건이 2,088,000 / 1,891,000 / 1,878,000으로 한 건이 지배하지 않았습니다.
- 남은 추가 검증: 완료율이 낮은 달(2025-09, 2026-01)의 취소·환불이 특정 결제수단이나 카테고리에 몰렸는지 월 × 결제수단 교차표로 확인합니다. 이번 데이터에는 프로모션·재고·배송 정보가 없어 "왜" 완료되지 않았는지는 확인할 수 없습니다.

### 사례 3. 카드 결제의 환불 비율
- 관찰: 기간 내 card 주문 71건 중 refunded 20건(28.2%), 다른 결제수단은 224건 중 36건(16.1%)입니다. 결제수단 4개 × 상태 3개 전체 카이제곱은 p=0.099입니다.
- 가설: 카드 결제는 환불 처리가 쉬워 환불 비율이 높을 수 있다. 또는 카드 결제 주문에 환불이 잦은 상품이 몰렸을 수 있다.
- 추가 검증 (실행함): card 대 나머지의 Fisher 정확 검정은 p=0.036이었습니다. 하지만 이 비교는 결과를 본 뒤 고른 비교라 우연히 유의하게 나올 가능성이 큽니다(결제수단 4개를 하나씩 비교하면 비교 횟수만큼 그럴 가능성이 늘어남). 전체 검정이 p=0.099이므로 "카드가 환불이 많다"고 결론 내리지 않았습니다. 같은 이유로 패션의 환불 금액 비율 32.9%도 확인했습니다. 패션 환불 금액 21,220,000 중 15,500,000이 주문 295 한 건이었고, 나머지는 5,720,000입니다. 주문 상세 행 기준 카테고리 × 상태 검정도 p=0.938이라 카테고리별 환불 차이는 없다고 봤습니다.
- 남은 추가 검증: 다음 기간 데이터가 쌓이면 card 환불 비율을 같은 기준으로 다시 계산합니다(사전에 정한 가설로 재검증). 환불 사유 컬럼이 있다면 사유별로 나눠 봅니다.

## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [41]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[WindowsPath('D:/dev/llm-data-analysis-course/reports/ch06_category_sales.csv'),
 WindowsPath('D:/dev/llm-data-analysis-course/reports/ch06_customer_city.csv'),
 WindowsPath('D:/dev/llm-data-analysis-course/reports/ch06_customer_sales.csv'),
 WindowsPath('D:/dev/llm-data-analysis-course/reports/ch06_eda_questions.csv'),
 WindowsPath('D:/dev/llm-data-analysis-course/reports/ch06_monthly_sales.csv'),
 WindowsPath('D:/dev/llm-data-analysis-course/reports/ch06_product_category.csv')]

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [42]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: D:\dev\llm-data-analysis-course\reports\ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [43]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [44]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
9,패션,545.0,64890000.0,26.39
6,식품,394.0,47044000.0,19.13
8,전자기기,303.0,37542000.0,15.27
2,뷰티,282.0,28614000.0,11.64
0,도서,277.0,28246000.0,11.49


In [45]:
# [추가 검증] src.eda 결과 == 공식 Notebook 셀 결과 ≠ 내가 정한 completed 기준
same_cat = module_results['category_sales'].reset_index(drop=True).equals(category_sales.reset_index(drop=True))
same_month = module_results['monthly_sales'].reset_index(drop=True).equals(monthly_sales.reset_index(drop=True))
print('src.eda category_sales == 공식 셀:', same_cat, '| monthly_sales == 공식 셀:', same_month)
print('src.eda category 합계:', f"{module_results['category_sales']['total_sales'].sum():,.0f}", '/ completed 기준 합계:', f'{BASE_TOTAL:,.0f}')

src.eda category_sales == 공식 셀: True | monthly_sales == 공식 셀: True
src.eda category 합계: 245,916,000 / completed 기준 합계: 138,528,000


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


### STEP 7. 재현 가능한 EDA 결과 확인

가이드 명령 `python scripts/run_eda.py`를 그대로 실행한 결과와 `PYTHONPATH`를 지정한 결과를 남겼습니다. 이어서 스크립트가 만든 `reports/ch06_*.csv`가 이 Notebook의 completed 기준 집계와 같은 범위인지 비교했습니다. 같은 명령을 PowerShell 터미널에서도 실행했습니다(`images/step05_reproducible.png`).

In [46]:
# [추가 검증] STEP 7. python scripts/run_eda.py 재실행 → reports 결과와 Notebook 범위 비교
env = {k: v for k, v in os.environ.items() if k != 'PYTHONPATH'}
env['PYTHONIOENCODING'] = 'utf-8'
run1 = subprocess.run([sys.executable, 'scripts/run_eda.py'], cwd=PROJECT_ROOT, env=env, capture_output=True, text=True, encoding='utf-8')
print('[1차] python scripts/run_eda.py → returncode', run1.returncode, '|', (run1.stderr.strip().splitlines() or [''])[-1])
run2 = subprocess.run([sys.executable, 'scripts/run_eda.py'], cwd=PROJECT_ROOT, env={**env, 'PYTHONPATH': str(PROJECT_ROOT)},
                      capture_output=True, text=True, encoding='utf-8')
print('[2차] PYTHONPATH=<프로젝트 루트> → returncode', run2.returncode)
print('\n'.join(run2.stdout.splitlines()[:12]))

rep_cat = pd.read_csv(REPORT_DIR / 'ch06_category_sales.csv')
rep_month = pd.read_csv(REPORT_DIR / 'ch06_monthly_sales.csv')
rep_cust = pd.read_csv(REPORT_DIR / 'ch06_customer_sales.csv')
scope_cmp = pd.DataFrame([
    {'결과': 'category_sales', 'reports/ 합계': rep_cat['total_sales'].sum(), '행': len(rep_cat), 'Notebook completed 기준 합계': q1['amount'].sum(), '행(기준)': len(q1)},
    {'결과': 'monthly_sales', 'reports/ 합계': rep_month['total_sales'].sum(), '행': len(rep_month), 'Notebook completed 기준 합계': q2['amount'].sum(), '행(기준)': len(q2)},
])
display(scope_cmp)
print('reports/ch06_category_sales.csv == 공식 Notebook 셀 category_sales:', np.isclose(rep_cat['total_sales'].sum(), category_sales['total_sales'].sum()))
print('reports/ch06_customer_sales.csv에 name 컬럼:', 'name' in rep_cust.columns)

[1차] python scripts/run_eda.py → returncode 1 | ModuleNotFoundError: No module named 'src'


[2차] PYTHONPATH=<프로젝트 루트> → returncode 0
6장 EDA 완료

[카테고리별 매출 상위 5개]
category  total_quantity  total_sales  sales_ratio
      패션           545.0   64890000.0        26.39
      식품           394.0   47044000.0        19.13
    전자기기           303.0   37542000.0        15.27
      뷰티           282.0   28614000.0        11.64
      도서           277.0   28246000.0        11.49

[월별 매출]
order_month  total_sales  order_count  avg_order_value


,결과,reports/ 합계,행,Notebook completed 기준 합계,행(기준)
0,category_sales,245916000.0,10,138528000.0,8
1,monthly_sales,248630000.0,16,138528000.0,14


reports/ch06_category_sales.csv == 공식 Notebook 셀 category_sales: True
reports/ch06_customer_sales.csv에 name 컬럼: True


### 재현 결과 해석
- 실행: 가이드 명령 그대로는 `ModuleNotFoundError: No module named 'src'`로 실패했고(Chapter 05와 같은 원인), 프로젝트 루트를 `PYTHONPATH`에 넣자 성공했습니다.
- 재현성: `src.eda` 결과는 공식 Notebook 셀 결과와 완전히 같습니다(`equals` True). 스크립트는 공식 셀을 정확히 재현합니다.
- 범위 비교: `reports/ch06_category_sales.csv` 합계는 245,916,000(10행, 표기 변형 포함, 주문 상태 미구분, 카테고리 없는 행 제외)입니다. `ch06_monthly_sales.csv`는 248,630,000(16개월, 2027-01·2028-12 포함)입니다. 이 Notebook의 completed 기준 합계 138,528,000과 범위가 다릅니다.
- 판단: 스크립트 결과는 "재현은 되지만 질문과 범위가 다른 결과"입니다. 재현 가능하다는 것과 맞는 범위라는 것은 별개입니다. Chapter 07에서는 `reports/ch06_*.csv` 대신 이 Notebook의 기준 데이터 집계를 사용하겠습니다.
- 개인정보: `run_eda.py`가 저장한 `reports/ch06_customer_sales.csv`에는 `name` 컬럼이 들어갑니다(`src/eda.py`가 이름이 있으면 함께 묶음). 이 파일은 강사 저장소의 실행 산출물이라 개인 저장소에 올리지 않았고, Notebook의 고객 집계 셀은 `[수정]`으로 이름을 빼고 실행했습니다.

## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## STEP 6. LLM으로 다음 질문 확장

### Safe Context (LLM에 준 정보)
원본 행과 고객 이름·ID는 넣지 않고, 컬럼 구조와 이 Notebook의 집계 결과만 넣었습니다.

```text
[데이터 구조] 온라인 쇼핑몰, 주문일 2025-07-01~2026-08-30
- orders: order_id, customer_id, order_date, payment_method(4종), order_status(completed/cancelled/refunded)
- order_items: order_id, product_id, quantity, unit_price, line_total(= quantity × unit_price)
- products: product_id, product_name, category(7종), price
- customers: customer_id, gender, age, city, signup_date
- 할인·쿠폰·배송·재고·환불 사유 컬럼은 없음

[집계 요약] completed 기준 상품 주문 금액 138,528,000 (주문 176건)
- 카테고리: 패션 23.8%, 식품 17.6%, 전자기기 14.8%, 뷰티 12.7%, 도서 12.2%, 스포츠 9.6%, 생활용품 6.6%
- 월별 완료율 33.3%~77.3%, 월별 금액과 완료율 상관 0.60, 주문당 금액과 상관 0.10
- 결제수단별 환불 비율: card 28.2%, 나머지 15~17% (전체 카이제곱 p=0.099)
```

### Prompt
```text
위 데이터 구조와 집계 요약으로 다음 EDA 질문 후보를 6개 제안해 주세요.
조건:
- 각 질문에 필요한 테이블·컬럼과 계산 지표를 함께 적을 것
- 데이터에 없는 컬럼이 필요하면 "현재 데이터로 불가"라고 표시할 것
- 원인을 단정하지 말고 질문 형태로 쓸 것
```

### LLM 제안 요약
1. 고객의 첫 구매 후 재구매까지 걸린 기간은 얼마인가? (orders.customer_id, order_date)
2. 가입 후 30일 이내에 첫 주문한 신규 고객의 주문 비중은? (customers.signup_date, orders.order_date)
3. 할인·쿠폰을 쓴 주문은 완료율이 다른가? (현재 데이터로 불가라고 표시)
4. 요일별 completed 주문 수와 금액은 다른가? (orders.order_date)
5. 도시별 주문당 금액은 다른가? (customers.city, order_items.line_total)
6. 배송 지연이 환불로 이어지는가? (현재 데이터로 불가라고 표시)

아래 셀에서 6개 제안을 실제 데이터로 계산해 가능 여부를 확인했습니다.

In [47]:
# [추가 검증] LLM 제안 질문이 실제 데이터로 계산 가능한지 하나씩 확인
llm_checks = []

# 1) 첫 구매 후 재구매까지 걸린 기간
cust_orders = base.groupby('customer_id')['order_date'].agg(['min', 'max', 'nunique'])
repeat = cust_orders[cust_orders['nunique'] >= 2]
gap = (repeat['max'] - repeat['min']).dt.days
llm_checks.append(('재구매까지 기간', f'가능: completed 재구매 고객 {len(repeat)}명, 첫~마지막 주문 간격 중앙값 {gap.median():.0f}일'))

# 2) 신규 고객(가입 30일 이내) 주문 비중
sign = customers.set_index('customer_id')['signup_date']
dd = (base.drop_duplicates('order_id')['order_date'] - base.drop_duplicates('order_id')['customer_id'].map(sign)).dt.days
llm_checks.append(('가입 후 30일 이내 주문 비중', f'수정 필요: 주문일이 가입일보다 앞선 completed 주문 {int((dd < 0).sum())}건 / {int(dd.notna().sum())}건'))

# 3) 할인율·쿠폰 효과
llm_checks.append(('할인·쿠폰 효과', '불가: 할인/쿠폰 컬럼 없음 (unit_price == products.price 726행 일치, Chapter 05)'))

# 4) 요일별 주문
dow = base.drop_duplicates('order_id')['order_date'].dt.day_name().value_counts()
llm_checks.append(('요일별 completed 주문 수', f'가능: 최다 {dow.idxmax()} {dow.max()}건, 최소 {dow.idxmin()} {dow.min()}건'))

# 5) 지역별 객단가
city_aov = base.merge(customers[['customer_id', 'city']], on='customer_id', how='inner').groupby('city').agg(
    orders=('order_id', 'nunique'), amount=('line_total', 'sum'))
llm_checks.append(('도시별 주문당 금액', f'수정 후 사용: 주문 수 1~30건으로 편차 큼 (최소 {city_aov["orders"].min()}건)'))

# 6) 재고·배송 지연
llm_checks.append(('배송 지연과 환불 관계', '불가: 배송·재고 컬럼 없음'))

display(pd.DataFrame(llm_checks, columns=['LLM 제안 질문', '실제 데이터 확인 결과']))

,LLM 제안 질문,실제 데이터 확인 결과
0,재구매까지 기간,"가능: completed 재구매 고객 53명, 첫~마지막 주문 간격 중앙값 159일"
1,가입 후 30일 이내 주문 비중,수정 필요: 주문일이 가입일보다 앞선 completed 주문 32건 / 169건
2,할인·쿠폰 효과,"불가: 할인/쿠폰 컬럼 없음 (unit_price == products.price 726행 일치, Chapter 05)"
3,요일별 completed 주문 수,"가능: 최다 Wednesday 34건, 최소 Saturday 14건"
4,도시별 주문당 금액,수정 후 사용: 주문 수 1~30건으로 편차 큼 (최소 1건)
5,배송 지연과 환불 관계,불가: 배송·재고 컬럼 없음


## 답안 4. LLM 질문 확장

- Safe Context: 컬럼 구조, 없는 컬럼 목록, 이 Notebook의 집계 요약(합계·비중·완료율·검정 결과). 원본 행, 고객 이름·ID는 넣지 않음
- Prompt: 위 STEP 6의 Prompt (질문 6개, 필요한 컬럼·지표 명시, 불가한 질문 표시, 원인 단정 금지)
- LLM 제안 요약: 재구매 기간, 신규 고객 비중, 할인 효과, 요일별 주문, 도시별 주문당 금액, 배송 지연과 환불
- 실제 사용 여부와 판단 이유:

| LLM 제안 | 데이터 확인 결과 | 사용 여부 | 판단 이유 |
| --- | --- | --- | --- |
| 1 재구매까지 기간 | completed 재구매 고객 53명, 첫~마지막 주문 간격 중앙값 159일 | 사용 | 필요한 컬럼이 모두 있고 표본(53명)이 있음 |
| 2 가입 후 30일 이내 주문 | completed 주문 169건 중 32건이 가입일보다 앞섬 | 수정 후 사용 | `signup_date` 정의가 불명확(Chapter 05). "가입일 이후 주문만"으로 범위를 좁히고 제외 건수를 함께 보고 |
| 3 할인·쿠폰 효과 | 할인 컬럼 없음. unit_price가 마스터 가격과 726행 모두 같음 | 보류 | LLM도 불가라고 표시함. 데이터가 생기면 다시 검토 |
| 4 요일별 주문 | 수요일 34건 최다, 토요일 14건 최소 | 사용 | 계산 가능. 다만 요일당 주문 14~34건이라 차이 해석은 신중히 |
| 5 도시별 주문당 금액 | 도시별 completed 주문 1~30건 | 수정 후 사용 | 주문 5건 미만 도시(대전 3건, Unknown 1건)는 "표본 부족"으로 따로 묶고 비교 |
| 6 배송 지연과 환불 | 배송 컬럼 없음 | 보류 | 데이터에 없는 원인 |

- 사람이 수정한 내용:
  - 2번: LLM은 `signup_date`를 가입일로 그대로 믿었지만, 실제로는 주문일보다 늦은 가입일이 19%(32/169) 있었습니다. 그래서 범위를 "가입일 ≤ 주문일"로 좁히는 조건을 추가했습니다.
  - 5번: LLM은 도시별 평균만 제안했지만, 도시별 주문 수가 1~30건이라 그대로 비교하면 1건짜리 도시가 극단값이 됩니다. 최소 주문 수 기준을 추가했습니다.
  - 전체: LLM 제안에는 Q3 결과(p=0.099)를 이어서 확인할 질문이 없었습니다. "완료율이 낮은 달의 취소·환불이 특정 결제수단에 몰렸는가"를 다음 우선순위에 직접 넣었습니다.

![LLM EDA 질문 확장](images/step04_llm.png)

- Evidence: 위 `[추가 검증]` LLM 제안 확인 셀 Output

## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [48]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
# (completed + 분석 기간 기준. 고객이 없는 주문은 inner로 제외되며 그 금액을 따로 출력)
city_customers = customers['city'].value_counts().rename('customers')
city_amount = base.merge(customers[['customer_id', 'city']], on='customer_id', how='inner').groupby('city').agg(
    buying_customers=('customer_id', 'nunique'), orders=('order_id', 'nunique'), amount=('line_total', 'sum'))
task1 = city_amount.join(city_customers, how='outer').fillna(0).sort_values('amount', ascending=False)
task1['amount_per_customer'] = (task1['amount'] / task1['customers']).round(0)
task1['customer_share(%)'] = (task1['customers'] / task1['customers'].sum() * 100).round(1)
task1['amount_share(%)'] = (task1['amount'] / task1['amount'].sum() * 100).round(1)
display(task1)
print('도시 합계 + 고객 미연결:', f"{task1['amount'].sum() + base.loc[~cust_linked, 'line_total'].sum():,.0f}", '/ 기준', f'{BASE_TOTAL:,.0f}')

,buying_customers,orders,amount,customers,amount_per_customer,customer_share(%),amount_share(%)
city,,,,,,,
부산,15,30,23140000.0,23,1006087.0,15.3,16.7
서울,11,20,22211000.0,18,1233944.0,12.0,16.1
광주,16,28,19866000.0,21,946000.0,14.0,14.4
인천,12,23,15723000.0,16,982688.0,10.7,11.4
대구,12,18,14781000.0,21,703857.0,14.0,10.7
수원,8,13,12220000.0,10,1222000.0,6.7,8.8
성남,5,13,11288000.0,9,1254222.0,6.0,8.2
울산,8,16,10509000.0,17,618176.0,11.3,7.6
고양,6,10,7222000.0,8,902750.0,5.3,5.2


도시 합계 + 고객 미연결: 138,528,000 / 기준 138,528,000


In [49]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요. (completed 상품 주문 금액 기준)
task2 = q1.drop(index='(상품 정보 없음)')[['catalog_products', 'sold_products', 'amount']].copy()
task2['amount_per_catalog_product'] = (task2['amount'] / task2['catalog_products']).round(0)
task2['rank_products'] = task2['catalog_products'].rank(ascending=False, method='min').astype(int)
task2['rank_amount'] = task2['amount'].rank(ascending=False).astype(int)
display(task2.sort_values('amount', ascending=False))
print('상품 수 vs 금액 상관계수:', round(task2['catalog_products'].corr(task2['amount']), 3))

,catalog_products,sold_products,amount,amount_per_catalog_product,rank_products,rank_amount
category_clean,,,,,,
패션,16.0,15,32940000.0,2058750.0,2,1
식품,16.0,16,24345000.0,1521562.0,2,2
전자기기,14.0,14,20445000.0,1460357.0,4,3
뷰티,17.0,17,17542000.0,1031882.0,1,4
도서,13.0,13,16895000.0,1299615.0,5,5
스포츠,10.0,10,13237000.0,1323700.0,6,6
생활용품,10.0,10,9109000.0,910900.0,6,7


상품 수 vs 금액 상관계수: 0.742


In [50]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요. (completed 주문 단위)
order_level = base.groupby(['order_id', 'payment_method'], as_index=False)['line_total'].sum()
task3 = order_level.groupby('payment_method')['line_total'].agg(orders='count', mean='mean', median='median', max='max').round(0)
display(task3)
print('주문 수 합:', int(task3['orders'].sum()), '/ 기준 주문', base['order_id'].nunique())

,orders,mean,median,max
payment_method,,,,
bank_transfer,48,803167.0,746500.0,2082000.0
card,40,938550.0,693000.0,5483000.0
kakao_pay,49,734571.0,608000.0,2300000.0
naver_pay,39,677949.0,518000.0,1980000.0


주문 수 합: 176 / 기준 주문 176


In [51]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요. (completed 상품 주문 금액, 단위: 천 원)
task4 = base.pivot_table(index='order_month', columns='category_clean', values='line_total', aggfunc='sum', fill_value=0)
task4['합계'] = task4.sum(axis=1)
display((task4 / 1000).astype(int))
print('피벗 합계 == 월별 합계:', bool((task4['합계'] == q2['amount']).all()), '| 전체', f"{task4['합계'].sum():,.0f}")

category_clean,(상품 정보 없음),도서,뷰티,생활용품,스포츠,식품,전자기기,패션,합계
order_month,,,,,,,,,
2025-07,82,1040,1133,406,295,2123,1319,3163,9561
2025-08,470,1144,1050,983,417,505,2547,2246,9362
2025-09,780,380,125,735,438,528,1157,1156,5299
2025-10,82,1570,661,1334,1426,3088,1781,2687,12629
2025-11,420,583,1581,528,1271,2221,1035,1944,9583
2025-12,468,34,884,489,791,2048,1408,2459,8581
2026-01,0,74,376,804,2359,2465,590,1467,8135
2026-02,0,1827,275,646,1009,1193,1562,1177,7689
2026-03,140,1075,728,233,364,2792,1378,537,7247


피벗 합계 == 월별 합계: True | 전체 138,528,000


In [52]:
# 과제 5. 고객별 order_count와 avg_order_value로 반복 구매 / 고액 단발 구매 구분 (completed + 분석 기간)
cust = base[cust_linked].groupby('customer_id').agg(order_count=('order_id', 'nunique'), amount=('line_total', 'sum'))
cust['avg_order_value'] = (cust['amount'] / cust['order_count']).round(0)
aov_q75 = cust['avg_order_value'].quantile(0.75)
cust['segment'] = np.select([cust['order_count'] >= 2, cust['avg_order_value'] >= aov_q75],
                            ['반복 구매(2회 이상)', '고액 단발(1회, 주문당 금액 상위 25%)'], default='일반 단발')
seg = cust.groupby('segment').agg(customers=('amount', 'size'), amount=('amount', 'sum'), avg_order_value=('avg_order_value', 'median'))
seg['amount_share(%)'] = (seg['amount'] / seg['amount'].sum() * 100).round(1)
display(seg)
print(f'주문당 금액 75% 분위수 기준: {aov_q75:,.0f} | 주문 횟수 분포:', cust['order_count'].value_counts().sort_index().to_dict())

,customers,amount,avg_order_value,amount_share(%)
segment,,,,
"고액 단발(1회, 주문당 금액 상위 25%)",10,13734000.0,1378500.0,9.9
반복 구매(2회 이상),53,107831000.0,777500.0,78.1
일반 단발,33,16587000.0,528000.0,12.0


주문당 금액 75% 분위수 기준: 1,013,500 | 주문 횟수 분포: {1: 43, 2: 37, 3: 8, 4: 7, 6: 1}


### 22번 실습 과제 답안 (금액은 모두 completed + 분석 기간 기준 상품 주문 금액)

- 과제 1: 고객 수 1위는 부산(23명)이고, 금액도 부산이 1위(23,140,000)입니다. 하지만 대구는 고객 수 공동 2위(21명)인데 금액은 5위, 울산은 고객 17명으로 5위인데 금액은 8위입니다. 고객 1인당 금액은 성남(1,254,222)과 서울(1,233,944)이 높습니다. 고객 수와 금액 순위는 같지 않습니다. 도시 합계 + 고객 미연결 금액 = 138,528,000으로 일치합니다.
- 과제 2: 상품 수와 금액의 상관계수는 0.742입니다. 상품이 가장 많은 뷰티(17개)는 금액 4위이고, 패션(16개)은 1위입니다. 상품 수가 많을수록 금액이 높은 경향은 있지만 순위가 그대로 따라가지는 않습니다. 상품 1개당 금액은 패션 2,058,750으로 가장 높고 생활용품 910,900으로 가장 낮습니다.
- 과제 3: completed 주문당 금액 평균은 card 938,550 > bank_transfer 803,167 > kakao_pay 734,571 > naver_pay 677,949입니다. card 평균은 card로 결제된 주문 290(수량 50 행 포함, 주문 합계 5,483,000)의 영향이 큽니다. 중앙값으로 보면 bank_transfer 746,500 > card 693,000으로 순위가 바뀝니다.
- 과제 4: 월×카테고리 피벗 합계가 월별 합계와 14개월 모두 같습니다. 2026-04 패션(5,306,000)은 주문 290의 4,800,000이 대부분입니다.
- 과제 5: 기준은 반복 구매 = completed 주문 2회 이상, 고액 단발 = 1회 구매이면서 주문당 금액이 상위 25%(1,013,500 이상)로 정했습니다. 반복 구매 고객 53명이 금액의 78.1%, 고액 단발 10명이 9.9%, 일반 단발 33명이 12.0%입니다. 반복 구매 고객의 주문당 금액 중앙값(777,500)은 고액 단발(1,378,500)보다 낮습니다. 반복 구매 고객은 "자주, 적당히", 고액 단발 고객은 "한 번, 크게" 사는 그룹으로 구분됩니다.
- 과제 6: 현재 데이터로 답할 수 없는 질문을 바꾸는 Prompt입니다.

```text
다음 질문은 현재 데이터로 답할 수 없습니다: "고객들은 왜 카드 결제 주문을 더 많이 환불하나요?"
데이터 구조: (위 Safe Context와 동일. 환불 사유·고객 문의 컬럼 없음)
조건:
1. 이 질문에 답하려면 어떤 컬럼이 더 필요한지 먼저 적어 주세요.
2. 현재 컬럼만으로 계산 가능한 질문 3개로 바꿔 주세요. (예: 결제수단별 환불 비율이 월·카테고리를 통제해도 유지되는가?)
3. 각 질문의 지표, 필요한 테이블·컬럼, 표본 크기 주의점을 함께 적어 주세요.
4. 원인을 단정하는 문장은 쓰지 마세요.
```

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.


## 답안 5. 다음 분석 우선순위

1. 완료율이 낮은 달(2025-09 33.3%, 2026-01 34.6%)의 cancelled·refunded 주문이 특정 결제수단·카테고리·고객에 몰려 있는가? (월 × 결제수단 × 상태 교차표, 주문 수 기준)
2. 패션의 판매 수량이 많은 이유는 상품 집중인가, 주문당 수량인가? (패션 상품별 수량 상위 5개 비중, 카테고리별 주문당 평균 수량)
3. 반복 구매 고객 53명의 재구매 간격(중앙값 159일)과 카테고리 구성은 단발 고객과 다른가? (고객 연결 필요, `signup_date`는 사용하지 않음)

### 가장 먼저 확인하고 싶은 이유
- Q2에서 월별 상품 주문 금액과 가장 크게 같이 움직인 것이 완료율이었습니다(상관 0.599, 전체 주문 수는 0.295). 상관은 원인을 뜻하지 않으므로, 완료되지 않은 주문이 어디에 있는지 나눠 봐야 다음 판단을 할 수 있습니다.
- Q3의 결제수단 차이는 p=0.099로 결론을 내리지 못했습니다. 월로 나눠 보면 특정 기간에 몰린 현상인지, 전체에 고르게 있는 현상인지 구분할 수 있습니다.
- 업무적으로도 바로 확인할 가치가 있습니다. "2026-01에 주문 26건 중 17건이 취소·환불된 이유"는 결제·CS 담당자에게 물어볼 수 있는 구체적인 질문입니다.

### 현재 데이터만으로 단정할 수 없는 것
- 취소·환불의 원인: 사유, 배송, 재고, 고객 문의 정보가 없습니다.
- 카드 결제와 환불의 관계: 전체 검정 p=0.099이고, 카드만 따로 본 비교는 결과를 보고 고른 비교라 우연일 수 있습니다.
- 대량 주문(290 수량 50, 295 수량 100)의 성격: 기업 구매인지 입력 오류인지 알 수 없습니다.
- 월별 추세·계절성: 14개월뿐이라 같은 달이 두 번 있는 것은 7월(2025-07, 2026-07)과 8월(2025-08, 2026-08)뿐입니다. 계절성을 판단하기에는 반복이 부족합니다.
- 신규 고객 분석: `signup_date`가 주문일보다 늦은 주문이 있어 가입일 기준 분석은 정의 확인 전까지 할 수 없습니다.
- 회계상 매출: `line_total`은 상품 주문 금액이며 할인·배송비·세금·실제 결제액이 반영되지 않았습니다.

### 업무적으로 확인할 가치가 있는 질문
- 2025-09, 2026-01에 취소·환불을 늘린 운영 이슈(품절, 배송 지연, 결제 오류)가 있었는가?
- 주문 290(수량 50), 295(수량 100, 환불)은 B2B 주문이었는가? 대량 주문을 일반 주문과 같은 지표로 봐야 하는가?
- 카드 결제 환불은 결제 취소와 구분되어 기록되는가? (상태값 정의 확인)

In [53]:
# [추가 검증] Chapter 06 완료 Evidence 자동 점검
evidence = {
    'Notebook 실행 (.venv 커널, pandas 3.0.5)': sys.executable.endswith(os.path.join('.venv', 'Scripts', 'python.exe')) and pd.__version__ == '3.0.5',
    '전처리 데이터 = Chapter 05 공식 결과': changed == '',
    'EDA 질문 3개 이상 (질문·범위·지표·검증 연결)': len(my_questions) >= 3,
    'completed + 분석 기간 범위 적용': set(base['order_status']) == {'completed'} and base['order_date'].between(VALID_START, VALID_END).all(),
    '집계 총합 검증 (8개 집계 모두 일치)': bool((total_check['결과'] == 'PASS').all()),
    '관찰/가설/추가 검증 계산 실행 (사례 3개)': all(v in globals() for v in ['q1_no_big', 'cmp', 'p_card']),
    'LLM 제안 데이터 검증': len(llm_checks) >= 5,
    'run_eda.py 재실행 (returncode 0)': run2.returncode == 0,
    'Notebook 출력·제출 파일에 name 없음': 'name' not in customer_sales.columns,
}
for item, ok in evidence.items():
    print(f'{"PASS" if ok else "FAIL"}  {item}')

PASS  Notebook 실행 (.venv 커널, pandas 3.0.5)
PASS  전처리 데이터 = Chapter 05 공식 결과
PASS  EDA 질문 3개 이상 (질문·범위·지표·검증 연결)
PASS  completed + 분석 기간 범위 적용
PASS  집계 총합 검증 (8개 집계 모두 일치)
PASS  관찰/가설/추가 검증 계산 실행 (사례 3개)
PASS  LLM 제안 데이터 검증
PASS  run_eda.py 재실행 (returncode 0)
PASS  Notebook 출력·제출 파일에 name 없음


## Chapter 06 완료 Evidence

- Notebook 전체 실행: PASS (`.venv` 커널, pandas 3.0.5, 강사 저장소 공식 clone, 오류 0)
- 전처리 데이터 준비: PASS (Chapter 05 공식 결과와 MD5·git 기준 동일)
- EDA 질문 3개 이상: PASS (Q1~Q3, 질문·범위·데이터·지표·계산·검증 연결)
- completed 범위 확인: PASS (completed + 2025-07-01~2026-08-30, 437행 / 176건 / 138,528,000)
- 집계 총합 검증: PASS (8개 집계 차이 0, 주문 수·수량 검증 포함)
- 관찰/가설/추가 검증 분리: PASS (사례 3개, 추가 검증 계산 실행)
- LLM 제안 검증: PASS (6개 제안을 데이터로 확인, 사용 2 / 수정 후 사용 2 / 보류 2)
- 재현 가능한 EDA: PASS (`run_eda.py` 재실행 성공, 공식 셀과 동일. 단, reports 결과는 상태 미구분 범위라 Notebook 기준과 다름)
- 결과 해석과 한계 작성: PASS

## 최종 체크

- [x] 질문과 지표가 연결되어 있습니다.
- [x] 집계 총합을 검증했습니다.
- [x] 관찰과 가설을 구분했습니다.
- [x] 다음 검증 질문을 작성했습니다.
- [x] LLM 제안을 검증했습니다.
- [x] 최종 Notebook URL을 제출합니다.